<a href="https://colab.research.google.com/github/ibarr123/BUS1182026/blob/main/GroupAssignmnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
!pip install langgraph langchain langchain-openai python-dotenv

In [26]:
import os
from google.colab import userdata
from google.colab.userdata import SecretNotFoundError

# Pull the key from Colab Secrets
try:
    api_key_value = userdata.get("OPENAI_API_KEY")
    os.environ["OPENAI_API_KEY"] = api_key_value
    print("API key loaded from Colab Secrets.")
    print("OPENAI_API_KEY exists:", "OPENAI_API_KEY" in os.environ)
except SecretNotFoundError:
    print("ERROR: Secret 'OPENAI_API_KEY' does not exist in Colab Secrets.")
    print("Please add your OpenAI API key to Colab's secret manager.")
    print("To do this, click the 'Secrets' icon (a closed padlock) on the left sidebar,")
    print("then click 'Add new secret' and enter 'OPENAI_API_KEY' as the name and your actual OpenAI API key as the value.")
    # Ensure OPENAI_API_KEY is not set if it wasn't found
    if "OPENAI_API_KEY" in os.environ:
        del os.environ["OPENAI_API_KEY"]

print("Current folder:", os.getcwd())
print("Files here:", os.listdir())

API key loaded from Colab Secrets.
OPENAI_API_KEY exists: True
Current folder: /content
Files here: ['.config', 'sample_data']


In [27]:
import os
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = api_key

print("API key loaded successfully:", bool(api_key))

API key loaded successfully: True


In [ ]:
# -----------------------------
# 1. Setup
# -----------------------------
import os
from typing import Literal, TypedDict
from typing_extensions import Annotated
from google.colab import userdata

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

# Load API key from Colab Secrets
api_key = userdata.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Missing OPENAI_API_KEY in Colab Secrets")

os.environ["OPENAI_API_KEY"] = api_key

llm = ChatOpenAI(model="gpt-5-nano")

# -----------------------------
# 2. Store Data (KEEP YOURS)
# -----------------------------
products = [
    {"name": "Wireless Earbuds", "price": 49, "desc": "Great for music and calls"},
    {"name": "Gaming Mouse", "price": 35, "desc": "High precision for gaming"},
    {"name": "Laptop Stand", "price": 25, "desc": "Ergonomic for studying"},
]

orders = {
    "1001": {"status": "Shipped", "eta": "2 days"},
    "1002": {"status": "Processing", "eta": "3-5 days"},
}

refund_policy = """
Returns allowed within 30 days.
Items must be unused.
Refund processed in 5–7 days.
"""

# -----------------------------
# 3. State
# -----------------------------
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]
    route: str

# -----------------------------
# 4. Router Agent
# -----------------------------
def router_node(state: ChatState):
    system = SystemMessage(
        content="Classify the request as: order_status, refund_policy, or product_recommendation. Only return the label."
    )

    user_msg = state["messages"][-1]
    response = llm.invoke([system, user_msg])

    route = response.content.strip().lower()

    if route not in ["order_status", "refund_policy", "product_recommendation"]:
        route = "product_recommendation"

    return {"route": route}

# -----------------------------
# 5. Order Agent
# -----------------------------
def order_node(state: ChatState):
    text = state["messages"][-1].content

    order_id = None
    for oid in orders:
        if oid in text:
            order_id = oid
            break

    if order_id:
        order = orders[order_id]
        reply = f"Order #{order_id} is {order['status']}. ETA: {order['eta']}."
    else:
        reply = "Please provide your order number."

    return {"messages": [AIMessage(content=reply)]}

# -----------------------------
# 6. Refund Agent
# -----------------------------
def refund_node(state: ChatState):
    system = SystemMessage(
        content=f"You are a support agent. Use this policy:\n{refund_policy}"
    )

    response = llm.invoke([system] + state["messages"])
    return {"messages": [AIMessage(content=response.content)]}

# -----------------------------
# 7. Product Agent
# -----------------------------
def product_node(state: ChatState):
    catalog = "\n".join([f"{p['name']} (${p['price']}): {p['desc']}" for p in products])

    system = SystemMessage(
        content=f"Recommend 1-2 products from this list:\n{catalog}"
    )

    response = llm.invoke([system] + state["messages"])
    return {"messages": [AIMessage(content=response.content)]}

# -----------------------------
# 8. Routing Logic
# -----------------------------
def route_decision(state: ChatState) -> Literal["order_status", "refund_policy", "product_recommendation"]:
    return state["route"]

# -----------------------------
# 9. Build Graph
# -----------------------------
builder = StateGraph(ChatState)

builder.add_node("router", router_node)
builder.add_node("order_status", order_node)
builder.add_node("refund_policy", refund_node)
builder.add_node("product_recommendation", product_node)

builder.add_edge(START, "router")

builder.add_conditional_edges(
    "router",
    route_decision,
    {
        "order_status": "order_status",
        "refund_policy": "refund_policy",
        "product_recommendation": "product_recommendation",
    },
)

builder.add_edge("order_status", END)
builder.add_edge("refund_policy", END)
builder.add_edge("product_recommendation", END)

memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

# -----------------------------
# 10. Chat Loop
# -----------------------------
def chat():
    print("TechNest Chatbot (Agentic AI)")
    config = {"configurable": {"thread_id": "user-1"}}

    while True:
        user_input = input("You: ")
        if user_input.lower() == "quit":
            break

        result = graph.invoke(
            {"messages": [HumanMessage(content=user_input)]},
            config=config
        )

        print("Bot:", result["messages"][-1].content)

chat()

TechNest Chatbot (Agentic AI)
